# What is the first day of the week?

Depends where you live. My calendar starts on Monday, but a lot of people I know treat Sunday as the start. I wanted to check which is actually more common — by number of places, and also by number of people, since counting tiny islands the same as India feels a bit unfair.

There's also Saturday and even Friday in the data, which I didn't expect.

Files I'm using:
- `first-day-of-week.csv` — first weekday for each territory (also has units + paper size)
- `population.csv` — population in 2020, in millions
- `four-regions.csv` — africa / americas / asia / europe

In [ ]:
# colab: uncomment this and upload the 3 csv files
# from google.colab import files
# uploaded = files.upload()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('first-day-of-week.csv')
pop = pd.read_csv('population.csv')
regions = pd.read_csv('four-regions.csv')

print(df.shape, pop.shape, regions.shape)
df.head()

In [ ]:
pop.head()

In [ ]:
regions.head()

In [ ]:
# just checking what's in here before I start grouping stuff
print(df.columns.tolist())
print(df['first_day'].unique())
print(df['units'].unique())
print(df['paper'].unique())

Join everything on `alpha3` so I don't have to merge again for every question. Using left join from the weekday file because that's the main list.

In [ ]:
merged = df.merge(pop, on='alpha3', how='left').merge(regions, on='alpha3', how='left')
merged.head()

## 1. How many territories start the week on Fri / Sat / Sun / Mon?

Pretty straightforward, just a count.

In [ ]:
counts = df['first_day'].value_counts()
print(counts)
print()
print('total territories:', counts.sum())
print(counts / counts.sum() * 100)

Monday wins easily if you count places: 186 vs 55 for Sunday. Saturday is 15, Friday is only 1. So most of the world *on a map* is Monday-first.

## 2. What about people though?

Need the population file for this. Population is in millions.

In [ ]:
people = merged.groupby('first_day')['population'].sum().sort_values(ascending=False)
print(people)
print()
print('total pop in the merge:', people.sum())
print(people / people.sum() * 100)

This is the part I thought was interesting. Monday has way more territories, but Sunday actually has slightly more people (~3844 million vs ~3600). China is Monday which keeps Monday in the game, but India + US + Indonesia + Pakistan + Brazil + Mexico are all Sunday.

Saturday is still a real chunk of people (~432 million), mostly because of places like Egypt / Iran / Iraq. Friday is basically just Maldives (0.5 million).

## 3. Does it depend on region?

Like, is Europe all Monday and the Americas all Sunday, or is it messier than that?

In [ ]:
by_region = merged.groupby(['four_regions', 'first_day']).size().unstack(fill_value=0)
by_region

In [ ]:
# same thing but as % of the region, easier to read
by_region.div(by_region.sum(axis=1), axis=0) * 100

Europe is basically Monday (46 Monday, 2 Sunday). Africa is also mostly Monday. Americas actually go Sunday (20 vs 15). Asia is the mixed one — Monday 27, Sunday 21, plus most of the Saturday countries, and the one Friday.

So if I had to say who "predominantly" starts on Sunday: the Americas. Monday: Europe, and Africa if you're counting countries. Asia you can't really pick one.

## 4. Same question but weighted by population

I added this because Q3 felt incomplete after seeing China vs India.

In [ ]:
pop_region = merged.groupby(['four_regions', 'first_day'])['population'].sum().unstack(fill_value=0)
pop_region

In [ ]:
pop_region.div(pop_region.sum(axis=1), axis=0) * 100

Yeah, Asia flips to Sunday once you weight by people. Americas become *very* Sunday (~87%) because of the US/Brazil/Mexico. Europe stays Monday. Africa stays Monday but Saturday is like 15% of the people there (Egypt etc).

## 5. Who are the Friday / Saturday countries?

Was curious which one is Friday.

In [ ]:
merged[merged['first_day'].isin(['fri', 'sat'])][['territory', 'first_day', 'four_regions', 'population']].sort_values(['first_day', 'territory'])

Friday = Maldives. Saturday is mostly North Africa + Middle East, which makes sense with the weekend being Friday-Saturday in a lot of those countries.

## 6. The csv also has units and paper size

Not sure if this is related to first day of the week but it's sitting there so I looked.

In [ ]:
print(pd.crosstab(merged['units'], merged['first_day']))
print()
merged[merged['units'] != 'metric'][['territory', 'units', 'paper', 'first_day']]

In [ ]:
print(pd.crosstab(merged['paper'], merged['first_day']))
print()
merged[merged['paper'] == 'US-Letter'][['territory', 'four_regions', 'first_day', 'units']]

Almost everyone is metric + A4. The US-Letter / US units countries are mostly Americas and they tend to start on Sunday. Not that surprising, kind of the same US-vs-everyone-else split. I wouldn't say units *cause* the weekday thing, they just travel together.

## 7. Did the merges actually match everything?

pop and regions have fewer rows than the weekday file so I figured some stuff got dropped / came in as NaN.

In [ ]:
print('rows in weekday file:', len(df))
print('missing population:', merged['population'].isna().sum())
print('missing region:', merged['four_regions'].isna().sum())

missing = merged[merged['population'].isna() | merged['four_regions'].isna()]
missing[['territory', 'alpha3', 'first_day', 'population', 'four_regions']].head(25)

61 territories have no pop / region. Looking at the names it's a lot of small islands, overseas territories, Antarctica, etc. So the "how many people" answers are really "how many people in the ~196 places Gapminder has", not literally every row in the CLDR file. Doesn't change the big picture much.

## 8. Biggest countries, just to sanity check Q2

In [ ]:
top = merged.sort_values('population', ascending=False).head(15)
top[['territory', 'four_regions', 'first_day', 'population']]

In [ ]:
top['first_day'].value_counts()

10 of the top 15 are Sunday, 4 Monday (China, Nigeria, Russia, Vietnam), 1 Saturday (Egypt). That tracks with Sunday being slightly ahead on population even though Monday dominates the country count.

## 9. Which region is actually split Sun vs Mon?

Ignoring Fri/Sat for a second and just looking at the two main ones.

In [ ]:
sun_mon = merged[merged['first_day'].isin(['sun', 'mon'])]
split = sun_mon.groupby(['four_regions', 'first_day'])['population'].sum().unstack()
split['sun_pct'] = split['sun'] / (split['sun'] + split['mon']) * 100
split

Asia is the split one (~60% Sunday). Americas are like 88% Sunday. Africa ~24% Sunday. Europe is 1% Sunday, so not split at all.

## 10. Quick summary table + charts

Putting the main numbers in one place.

In [ ]:
summary = merged.groupby('first_day').agg(
    n_territories=('territory', 'count'),
    n_with_pop=('population', 'count'),
    people=('population', 'sum')
)
summary['avg_pop'] = summary['people'] / summary['n_with_pop']
summary

Sunday countries are fewer but bigger on average, which is why the two views (map vs people) disagree.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

counts.plot(kind='bar', ax=ax[0], color='steelblue')
ax[0].set_title('territories')
ax[0].set_xlabel('first day')

people.plot(kind='bar', ax=ax[1], color='darkorange')
ax[1].set_title('population (millions)')
ax[1].set_xlabel('first day')

plt.show()

In [ ]:
by_region.plot(kind='bar', stacked=True, figsize=(8, 4))
plt.title('territories by region')
plt.xlabel('')
plt.show()

## What I got from this

If you count countries, the week starts on Monday. If you count people, Sunday and Monday are almost tied (Sunday a bit ahead). Europe is Monday, Americas are Sunday, Asia depends whether you count countries or people, Africa is Monday with a Saturday belt in the north. Friday is just Maldives.

Also the population/region files don't cover every territory in the weekday file, so the people numbers are a little incomplete, but the missing ones are mostly small so I don't think it changes the conclusion.